# MiniJEV 教程：从零开始训练 System One 决策模型

本教程将带你一步步完成一个类 JEV 的轻量级决策模型训练。

## 你将学到：
1. 什么是 System One 决策模型
2. 如何准备训练数据
3. 如何使用 LoRA 微调 Qwen3-0.6B
4. 如何进行推理和评估

## 1. 环境准备

首先安装必要的依赖包：

In [ ]:
# 安装依赖（如果在 Colab 中运行）
import sys
if 'google.colab' in str(get_ipython()):
    !pip install transformers peft datasets accelerate torch tqdm
    !git clone https://huggingface.co/Qwen/Qwen3-0.6B

In [ ]:
import torch
import json
import os
from tqdm import tqdm

# 检查设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 理解 System One 模型

### 传统 LLM (System Two) vs System One

| 特性 | System Two (ChatGPT等) | System One (JEV) |
|------|------------------------|------------------|
| 输出 | 完整文本 | 类型化答案 + 概率 |
| 延迟 | 秒级 | 毫秒级 |
| 适用场景 | 开放式生成 | 分类/路由/决策 |

### 核心原理

System One 模型**不做自回归解码**，而是直接读取模型对候选选项的 logit 值：

```python
# 伪代码
logits = model(inputs).logits[:, -1, :]  # 最后一个 token
option_a_logit = logits[0, tokenizer("A").input_ids[0]]
option_b_logit = logits[0, tokenizer("B").input_ids[0]]
probs = softmax([option_a_logit, option_b_logit])
```

## 3. 加载基座模型

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 加载 tokenizer
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 设置 pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(device)

print(f"Model loaded: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## 4. 体验 Logit 提取

让我们看看如何从模型中提取选项的概率：

In [ ]:
def get_option_probabilities(prompt, options):
    """计算各选项的概率分布"""
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs.input_ids.to(device)
    
    # 前向传播
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits[:, -1, :]  # 最后一个 token 的 logits
    
    # 提取选项对应的 logit
    option_logits = []
    for option in options:
        token_id = tokenizer.encode(option, add_special_tokens=False)[0]
        option_logits.append(logits[0, token_id].item())
    
    # Softmax 转概率
    import torch.nn.functional as F
    probs = F.softmax(torch.tensor(option_logits), dim=0)
    
    return {opt: prob.item() for opt, prob in zip(options, probs)}

# 测试
prompt = """判断以下问题：系统响应很慢，CPU占用100%。
最可能的原因是什么？
A. 性能问题
B. 配置错误
C. 代码缺陷
D. 网络问题

Answer:"""

options = ["A", "B", "C", "D"]
probs = get_option_probabilities(prompt, options)

print("选项概率分布:")
for opt, prob in sorted(probs.items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 30)
    print(f"  {opt}: {prob:.2%} {bar}")

## 5. 准备训练数据

JEV 式模型需要特殊的数据格式。每个样本包含：
- `state`: 上下文状态
- `questions`: 决策问题列表
- `answers`: 期望答案

In [ ]:
# 创建示例数据
sample_data = [
    {
        "id": "sample_001",
        "state": {"user_message": "系统响应很慢", "priority": "high"},
        "questions": [
            {"type": "choice", "text": "根因？", "options": ["性能问题", "配置错误", "代码缺陷"]}
        ],
        "answers": [{"question_id": "q1", "answer": "性能问题", "confidence": 0.85}]
    },
    {
        "id": "sample_002",
        "state": {"file_type": "java", "change_lines": 150},
        "questions": [
            {"type": "choice", "text": "风险等级？", "options": ["低", "中", "高"]}
        ],
        "answers": [{"question_id": "q1", "answer": "中", "confidence": 0.7}]
    }
]

print("示例数据:")
print(json.dumps(sample_data[0], indent=2, ensure_ascii=False))

## 6. 应用 LoRA 进行微调

LoRA (Low-Rank Adaptation) 只训练少量参数（约 0.1%），大幅降低显存需求。

In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA 配置
lora_config = LoraConfig(
    r=8,                      # 秩
    lora_alpha=32,            # 缩放因子
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用 LoRA
model = get_peft_model(model, lora_config)

# 查看可训练参数
model.print_trainable_parameters()

## 7. 总结

恭喜你完成了 MiniJEV 的基础教程！

### 你学到了：
1. ✅ System One 模型的原理
2. ✅ 如何从模型中提取 logit 概率
3. ✅ 使用 LoRA 进行参数高效微调

### 下一步：
- 准备更多训练数据
- 在 GPU 上运行完整训练
- 评估模型性能

参考完整代码：`scripts/train.py`